# Seizure-Prediction Metric Framework

**Task scope.** This notebook exists solely to define a reusable set of
seizure-prediction evaluation metrics. It is a **prerequisite** for the
first Logistic Regression LOSO baseline (`EEG_logistic_loso_binary.ipynb`),
which was blocked without it: plain accuracy/F1 on individually-labeled
epochs is exactly the metric that made the earlier 4-class TCN look
deceptively "successful" (78.9% accuracy while detecting essentially
nothing -- see `project_state.md` Section 18.1) by rewarding a classifier
that mostly predicts the majority class.

**No model is trained here.** This notebook only defines functions and
demonstrates them on tiny synthetic examples so their behaviour is
verified before any real model output is scored with them.

**Every metric definition below is a documented, revisable methodological
choice**, not an established literature standard reproduced verbatim --
they are flagged explicitly so they can be revisited before anything here
is used in a paper.

## What a fold's predictions must look like

Every function in this notebook operates on a **fold predictions
DataFrame**: one row per TEST epoch belonging to ONE (patient, horizon,
held-out seizure) LOSO fold, with these columns:

| column | meaning |
|---|---|
| `patient_id` | e.g. `chb12` |
| `horizon_minutes` | 5 / 15 / 30 |
| `fold_id`, `held_out_seizure_id` | which LOSO fold this epoch's prediction came from |
| `recording_id`, `epoch_idx` | chronological identity within the recording (epoch_idx increases by 1 for consecutive 30s epochs in the same recording) |
| `t_start_sec`, `t_end_sec` | epoch timing, recording-relative seconds |
| `time_to_next_seizure_sec` | seconds from this epoch's end to the next seizure onset (only meaningful/used for PREICTAL epochs) |
| `y_true` | 1 = PREICTAL, 0 = INTERICTAL |
| `y_prob` | predicted probability of PREICTAL (class 1) |
| `y_pred` | 1/0 decision at the configured threshold |


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix,
)

print("=" * 60)
print("SEIZURE-PREDICTION METRIC FRAMEWORK -- SETUP")
print("=" * 60)
print(f"numpy : {np.__version__}")
print(f"pandas: {pd.__version__}")

REQUIRED_PRED_COLS = [
    "patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id",
    "recording_id", "epoch_idx", "t_start_sec", "t_end_sec",
    "time_to_next_seizure_sec", "y_true", "y_prob", "y_pred",
]

# Epoch duration is a project-wide established constant (30s, non-overlapping
# epochs -- project_state.md Section 5.1). It is not hardcoded blindly here:
# any caller passing real data is expected to have already verified it from
# t_end_sec - t_start_sec; this notebook's own synthetic self-tests set it
# explicitly. Kept as a module-level default so a caller can override it.
EPOCH_SEC_DEFAULT = 30.0
print(f"[OK] Metric framework imports ready. EPOCH_SEC_DEFAULT={EPOCH_SEC_DEFAULT}")


SEIZURE-PREDICTION METRIC FRAMEWORK -- SETUP
numpy : 2.0.2
pandas: 2.3.3
[OK] Metric framework imports ready. EPOCH_SEC_DEFAULT=30.0


## Input validation

Fail loudly on a malformed predictions table rather than silently computing
metrics on it.


In [2]:

def validate_predictions_df(df, require_single_fold=False):
    missing = [c for c in REQUIRED_PRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"predictions_df missing required columns: {missing}")
    if df["y_true"].isna().any() or not set(df["y_true"].unique()) <= {0, 1}:
        raise ValueError("y_true must be exactly {0, 1} with no NaNs")
    if df["y_pred"].isna().any() or not set(df["y_pred"].unique()) <= {0, 1}:
        raise ValueError("y_pred must be exactly {0, 1} with no NaNs")
    if df["y_prob"].isna().any() or (df["y_prob"] < 0).any() or (df["y_prob"] > 1).any():
        raise ValueError("y_prob must be in [0, 1] with no NaNs")
    if require_single_fold:
        key_cols = ["patient_id", "horizon_minutes", "held_out_seizure_id"]
        if df[key_cols].drop_duplicates().shape[0] != 1:
            raise ValueError("fold_df must contain exactly one (patient, horizon, held-out seizure)")
    return True


print("[OK] validate_predictions_df() defined.")


[OK] validate_predictions_df() defined.


## Primary (event-level / seizure-prediction) metrics

These are the metrics the supervisor asked to be primary, because
epoch-level accuracy/F1 alone cannot distinguish "detects seizures" from
"mostly predicts the majority class" (project_state.md Section 18.1).

**Design choices, stated explicitly:**

1. **Seizure sensitivity** (per fold) = did the model predict PREICTAL for
   at least one TEST epoch that truly is PREICTAL for the held-out seizure?
   Binary per fold (a fold IS one held-out seizure by LOSO construction), so
   pooled sensitivity across folds = (seizures detected) / (eligible
   seizures) -- an event count ratio, not a per-epoch recall.
2. **Time in warning** (per fold) = `time_to_next_seizure_sec` of the
   *earliest* (i.e. largest `time_to_next_seizure_sec`) TEST PREICTAL epoch
   that was predicted PREICTAL. This is the advance notice the *first*
   correct alarm would have given if acted on immediately. It does **not**
   require the alarm to persist continuously afterward -- a simplifying
   choice for this first baseline, stated here so it can be revisited.
   Undefined (`NaN`) when the seizure is not detected at all -- never 0.
3. **False alarms** (per fold) = number of **alarm events** among TEST
   INTERICTAL epochs, where an alarm event is a maximal run of
   *consecutive* (by `epoch_idx`, i.e. temporally adjacent 30s epochs in the
   same recording) predicted-PREICTAL epochs. Consecutive positive epochs
   are merged into ONE event rather than counted individually, so a single
   sustained false alarm is not inflated into many. **False alarms/hour** =
   alarm events / TEST-INTERICTAL hours in that fold; `NaN` (not 0) when the
   fold has 0 TEST INTERICTAL epochs (10 of 19 LOSO folds -- see
   `project_state.md` Section 20.5), since a rate has no denominator there.


In [3]:

def _count_alarm_events(sorted_epoch_idx, sorted_y_pred):
    '''Number of maximal runs of consecutive predicted-positive epochs,
    where "consecutive" requires epoch_idx to increase by exactly 1
    (so a chronological gap always starts a new potential event, even if
    both epochs on either side of the gap were predicted positive).'''
    n = len(sorted_epoch_idx)
    if n == 0:
        return 0
    epoch_idx = np.asarray(sorted_epoch_idx)
    y_pred = np.asarray(sorted_y_pred)
    new_block = np.concatenate(([True], np.diff(epoch_idx) != 1))
    block_id = np.cumsum(new_block)
    n_events = 0
    for b in np.unique(block_id):
        p = y_pred[block_id == b]
        new_run = np.concatenate(([True], np.diff(p) != 0))
        run_id = np.cumsum(new_run)
        for r in np.unique(run_id):
            if p[run_id == r][0] == 1:
                n_events += 1
    return n_events


def fold_seizure_event_metrics(fold_df, epoch_sec=EPOCH_SEC_DEFAULT):
    '''Primary, event-level seizure-prediction metrics for ONE LOSO fold.'''
    validate_predictions_df(fold_df, require_single_fold=True)

    pre = fold_df[fold_df["y_true"] == 1].sort_values("t_start_sec")
    inter = fold_df[fold_df["y_true"] == 0].sort_values(["recording_id", "epoch_idx"])

    n_preictal_alarms = int((pre["y_pred"] == 1).sum())
    detected = int(n_preictal_alarms > 0)

    if detected:
        alarm_epochs = pre[pre["y_pred"] == 1]
        first_alarm = alarm_epochs.loc[alarm_epochs["time_to_next_seizure_sec"].idxmax()]
        time_in_warning_sec = float(first_alarm["time_to_next_seizure_sec"])
    else:
        time_in_warning_sec = np.nan

    n_false_alarm_events = _count_alarm_events(inter["epoch_idx"].to_numpy(), inter["y_pred"].to_numpy())
    n_interictal_epochs = len(inter)
    interictal_hours = n_interictal_epochs * epoch_sec / 3600.0
    false_alarms_per_hour = (
        n_false_alarm_events / interictal_hours if interictal_hours > 0 else np.nan
    )

    return {
        "n_test_preictal": len(pre),
        "n_test_interictal": n_interictal_epochs,
        "seizure_detected": detected,
        "n_preictal_alarm_epochs": n_preictal_alarms,
        "time_in_warning_sec": time_in_warning_sec,
        "n_false_alarm_events": n_false_alarm_events,
        "interictal_hours": interictal_hours,
        "false_alarms_per_hour": false_alarms_per_hour,
    }


print("[OK] fold_seizure_event_metrics() defined.")


[OK] fold_seizure_event_metrics() defined.


## Secondary (epoch-level classification) metrics

Standard sklearn metrics on the same TEST epochs, treated as secondary
context, never as the headline result. `zero_division=np.nan` is passed
explicitly everywhere so an undefined precision/recall/F1 (e.g. a fold with
0 predicted-positive epochs, or 0 true-negative epochs) surfaces as `NaN`,
never a silently-substituted 0 -- per the supervisor's explicit instruction
not to convert undefined values to 0. AUC-PR (`average_precision_score`) is
mathematically degenerate (trivially 1.0) when only one class is present in
`y_true` (e.g. the 10 folds with 0 TEST INTERICTAL epochs), so that case is
detected explicitly and reported as `NaN` rather than trusting sklearn's
degenerate value.


In [4]:

def fold_epoch_metrics(fold_df):
    '''Secondary, epoch-level classification metrics for ONE LOSO fold (or
    any pooled set of epochs -- this function does not require a single
    fold, unlike fold_seizure_event_metrics).'''
    validate_predictions_df(fold_df, require_single_fold=False)

    y_true = fold_df["y_true"].to_numpy()
    y_pred = fold_df["y_pred"].to_numpy()
    y_prob = fold_df["y_prob"].to_numpy()

    n_classes_true = len(np.unique(y_true))

    accuracy = float(accuracy_score(y_true, y_pred))
    precision = float(precision_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    recall = float(recall_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    f1 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=np.nan))

    if n_classes_true < 2:
        auc_pr = np.nan
        auc_pr_undefined_reason = "only one class present in y_true (AUC-PR is not meaningful)"
    else:
        auc_pr = float(average_precision_score(y_true, y_prob))
        auc_pr_undefined_reason = None

    return {
        "n_epochs": len(fold_df),
        "n_classes_in_test": n_classes_true,
        "accuracy": accuracy,
        "precision_preictal": precision,
        "recall_preictal": recall,
        "f1_preictal": f1,
        "auc_pr": auc_pr,
        "auc_pr_undefined_reason": auc_pr_undefined_reason,
    }


print("[OK] fold_epoch_metrics() defined.")


[OK] fold_epoch_metrics() defined.


## Pooled aggregation

Naively averaging per-fold *rates* is wrong here: many folds have 0 TEST
interictal epochs (undefined FA/hour) or a single class in TEST (undefined
AUC-PR/precision/recall), and folds vary hugely in TEST size (1 epoch to
dozens). Instead:

- **Pooled event-count metrics** (sensitivity, FA/hour) sum the underlying
  counts (detections, alarm events, interictal hours) across the included
  folds and divide once, so folds with 0 denominator contribute 0 to both
  numerator and denominator rather than an undefined ratio distorting a
  mean.
- **Pooled epoch-level metrics** (AUC-PR, F1, accuracy, precision, recall)
  concatenate the TEST epochs of the included folds into one set and
  compute a single classification metric on the union, rather than
  averaging per-fold values (most of which are undefined for many folds).
- **Time in warning** is only defined for *detected* seizures, so its
  pooled summary reports the distribution (mean/median/min/max) over
  detected folds only, plus how many of the eligible folds that covers.


In [5]:

def pool_epoch_metrics(pooled_epochs_df):
    '''Pooled epoch-level metrics over the concatenation of several folds'
    TEST epochs (e.g. all folds for one patient+horizon, or all folds for
    one horizon across both patients).'''
    return fold_epoch_metrics(pooled_epochs_df)


def pool_event_metrics(fold_event_metrics_df):
    '''Pooled event-level metrics over several folds' event-metric rows
    (one row per fold, as produced by fold_seizure_event_metrics()).'''
    n_folds = len(fold_event_metrics_df)
    n_detected = int(fold_event_metrics_df["seizure_detected"].sum())
    seizure_sensitivity = n_detected / n_folds if n_folds > 0 else np.nan

    total_fa_events = int(fold_event_metrics_df["n_false_alarm_events"].sum())
    total_interictal_hours = float(fold_event_metrics_df["interictal_hours"].sum())
    pooled_fa_per_hour = (
        total_fa_events / total_interictal_hours if total_interictal_hours > 0 else np.nan
    )

    detected_rows = fold_event_metrics_df[fold_event_metrics_df["seizure_detected"] == 1]
    tiw = detected_rows["time_in_warning_sec"].dropna()
    if len(tiw) > 0:
        tiw_mean, tiw_median, tiw_min, tiw_max = (
            float(tiw.mean()), float(tiw.median()), float(tiw.min()), float(tiw.max())
        )
    else:
        tiw_mean = tiw_median = tiw_min = tiw_max = np.nan

    return {
        "n_folds": n_folds,
        "n_seizures_detected": n_detected,
        "seizure_sensitivity": seizure_sensitivity,
        "total_false_alarm_events": total_fa_events,
        "total_interictal_hours": total_interictal_hours,
        "false_alarms_per_hour": pooled_fa_per_hour,
        "n_detected_with_time_in_warning": int(len(tiw)),
        "time_in_warning_sec_mean": tiw_mean,
        "time_in_warning_sec_median": tiw_median,
        "time_in_warning_sec_min": tiw_min,
        "time_in_warning_sec_max": tiw_max,
    }


print("[OK] pool_epoch_metrics() / pool_event_metrics() defined.")


[OK] pool_epoch_metrics() / pool_event_metrics() defined.


## Self-test on synthetic data

Cheap, deterministic verification that the alarm-event counting and
NaN/undefined handling behave as documented, before any real fold is scored
with these functions.


In [6]:

def _make_synthetic_fold(rows):
    '''rows: list of (t_start_sec, epoch_idx, y_true, y_pred, y_prob, time_to_next_seizure_sec)'''
    df = pd.DataFrame(rows, columns=[
        "t_start_sec", "epoch_idx", "y_true", "y_pred", "y_prob", "time_to_next_seizure_sec",
    ])
    df["t_end_sec"] = df["t_start_sec"] + EPOCH_SEC_DEFAULT
    df["patient_id"] = "synthetic_patient"
    df["horizon_minutes"] = 5
    df["fold_id"] = 1
    df["held_out_seizure_id"] = "synthetic_sz0"
    df["recording_id"] = "synthetic_rec"
    return df[REQUIRED_PRED_COLS]


print("Self-test 1: two separate false-alarm runs (gap between them) -> 2 events, not 1 or 4")
# 6 interictal epochs, predicted positive at idx {0,1} and {3,4} (a gap at idx 2), plus
# one detected preictal epoch.
rows = [
    (0, 0, 0, 1, 0.9, np.nan),
    (30, 1, 0, 1, 0.8, np.nan),
    (60, 2, 0, 0, 0.1, np.nan),
    (90, 3, 0, 1, 0.7, np.nan),
    (120, 4, 0, 1, 0.6, np.nan),
    (150, 5, 0, 0, 0.2, np.nan),
    (300, 6, 1, 1, 0.95, 120.0),
    (330, 7, 1, 0, 0.4, 90.0),
]
fold1 = _make_synthetic_fold(rows)
ev1 = fold_seizure_event_metrics(fold1)
assert ev1["n_false_alarm_events"] == 2, ev1
assert ev1["seizure_detected"] == 1
assert ev1["time_in_warning_sec"] == 120.0, "must use the EARLIEST correct alarm's dt, not the latest"
assert ev1["n_test_interictal"] == 6
assert np.isclose(ev1["interictal_hours"], 6 * EPOCH_SEC_DEFAULT / 3600.0)
assert np.isclose(ev1["false_alarms_per_hour"], 2 / ev1["interictal_hours"])
print("  [OK]", ev1)

print("\nSelf-test 2: seizure NOT detected -> time_in_warning_sec is NaN, not 0")
rows2 = [
    (0, 0, 0, 0, 0.1, np.nan),
    (300, 1, 1, 0, 0.3, 60.0),
    (330, 2, 1, 0, 0.2, 30.0),
]
fold2 = _make_synthetic_fold(rows2)
ev2 = fold_seizure_event_metrics(fold2)
assert ev2["seizure_detected"] == 0
assert np.isnan(ev2["time_in_warning_sec"])
print("  [OK]", ev2)

print("\nSelf-test 3: zero TEST interictal epochs -> false_alarms_per_hour is NaN, not 0")
rows3 = [
    (300, 0, 1, 1, 0.9, 60.0),
    (330, 1, 1, 0, 0.3, 30.0),
]
fold3 = _make_synthetic_fold(rows3)
ev3 = fold_seizure_event_metrics(fold3)
assert ev3["n_test_interictal"] == 0
assert np.isnan(ev3["false_alarms_per_hour"]), "must be NaN, not 0, per explicit instruction"
print("  [OK]", ev3)

print("\nSelf-test 4: single-class TEST (all positive) -> auc_pr is NaN, not a trivial 1.0")
epoch3 = fold_epoch_metrics(fold3)
assert epoch3["n_classes_in_test"] == 1
assert np.isnan(epoch3["auc_pr"]), "AUC-PR must be NaN when only one class is present"
print("  [OK]", epoch3)

print("\nSelf-test 5: 0 predicted-positive epochs -> precision is NaN (0/0), not 0")
rows5 = [
    (0, 0, 0, 0, 0.1, np.nan),
    (300, 1, 1, 0, 0.2, 60.0),
]
fold5 = _make_synthetic_fold(rows5)
epoch5 = fold_epoch_metrics(fold5)
assert np.isnan(epoch5["precision_preictal"]), epoch5
print("  [OK]", epoch5)

print("\nSelf-test 6: pool_event_metrics sums counts correctly across folds with mixed undefined values")
# ev1: detected, time_in_warning=120.0, 2 FA events, 6 interictal epochs
# ev2: NOT detected (time_in_warning excluded from the pooled average)
# ev3: detected, time_in_warning=60.0, 0 interictal epochs (0 FA events, 0 hours)
pooled_events = pd.DataFrame([ev1, ev2, ev3])
pooled = pool_event_metrics(pooled_events)
assert pooled["n_folds"] == 3
assert pooled["n_seizures_detected"] == 2  # ev1 and ev3 detected; ev2 not
assert np.isclose(pooled["seizure_sensitivity"], 2 / 3)
assert pooled["total_false_alarm_events"] == 2  # ev2 and ev3 contribute 0
expected_hours = ev1["interictal_hours"] + ev2["interictal_hours"] + ev3["interictal_hours"]
assert np.isclose(pooled["total_interictal_hours"], expected_hours)
assert np.isclose(pooled["false_alarms_per_hour"], 2 / expected_hours)
assert pooled["n_detected_with_time_in_warning"] == 2  # ev1 and ev3
assert np.isclose(pooled["time_in_warning_sec_mean"], (120.0 + 60.0) / 2)
print("  [OK]", pooled)

print("\n[ALL SELF-TESTS PASSED]")


Self-test 1: two separate false-alarm runs (gap between them) -> 2 events, not 1 or 4
  [OK] {'n_test_preictal': 2, 'n_test_interictal': 6, 'seizure_detected': 1, 'n_preictal_alarm_epochs': 1, 'time_in_warning_sec': 120.0, 'n_false_alarm_events': 2, 'interictal_hours': 0.05, 'false_alarms_per_hour': 40.0}

Self-test 2: seizure NOT detected -> time_in_warning_sec is NaN, not 0
  [OK] {'n_test_preictal': 2, 'n_test_interictal': 1, 'seizure_detected': 0, 'n_preictal_alarm_epochs': 0, 'time_in_warning_sec': nan, 'n_false_alarm_events': 0, 'interictal_hours': 0.008333333333333333, 'false_alarms_per_hour': 0.0}

Self-test 3: zero TEST interictal epochs -> false_alarms_per_hour is NaN, not 0


  [OK] {'n_test_preictal': 2, 'n_test_interictal': 0, 'seizure_detected': 1, 'n_preictal_alarm_epochs': 1, 'time_in_warning_sec': 60.0, 'n_false_alarm_events': 0, 'interictal_hours': 0.0, 'false_alarms_per_hour': nan}

Self-test 4: single-class TEST (all positive) -> auc_pr is NaN, not a trivial 1.0


  [OK]

 {'n_epochs': 2, 'n_classes_in_test': 1, 'accuracy': 0.5, 'precision_preictal': 1.0, 'recall_preictal': 0.5, 'f1_preictal': 0.6666666666666666, 'auc_pr': nan, 'auc_pr_undefined_reason': 'only one class present in y_true (AUC-PR is not meaningful)'}

Self-test 5: 0 predicted-positive epochs -> precision is NaN (0/0), not 0


  [OK]

{'n_epochs': 2, 'n_classes_in_test': 2, 'accuracy': 0.5, 'precision_preictal': nan, 'recall_preictal': 0.0, 'f1_preictal': 0.0, 'auc_pr': 1.0, 'auc_pr_undefined_reason': None}

Self-test 6: pool_event_metrics sums counts correctly across folds with mixed undefined values
  [OK] {'n_folds': 3, 'n_seizures_detected': 2, 'seizure_sensitivity': 0.6666666666666666, 'total_false_alarm_events': 2, 'total_interictal_hours': 0.058333333333333334, 'false_alarms_per_hour': 34.285714285714285, 'n_detected_with_time_in_warning': 2, 'time_in_warning_sec_mean': 90.0, 'time_in_warning_sec_median': 90.0, 'time_in_warning_sec_min': 60.0, 'time_in_warning_sec_max': 120.0}

[ALL SELF-TESTS PASSED]


## Explicitly NOT implemented here

- Any model, feature, or classifier -- this notebook is metrics-only.
- Threshold optimization (all functions take `y_pred` as already decided by
  the caller at whatever threshold it used; this notebook does not search
  for a better threshold).
- ROC-AUC, calibration metrics, or any metric beyond what the supervisor's
  task explicitly asked for (sensitivity, false alarms/hour, time in
  warning, AUC-PR, plus F1/accuracy/precision/recall as secondary).
- Statistical significance testing / confidence intervals on any pooled
  metric.

## Summary

Defined and self-tested: `fold_seizure_event_metrics()` (primary),
`fold_epoch_metrics()` (secondary), `pool_epoch_metrics()` /
`pool_event_metrics()` (pooled aggregation). `EEG_logistic_loso_binary.ipynb`
copies these functions verbatim (credited), the same reuse convention
`EEG_loso_evaluation_splitting.ipynb` already established for the
normalization functions.
